In [1]:
import pandas as pd
import pickle
from pymongo import MongoClient
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

print("--- TAHAP 1: AMBIL DATA DARI MONGODB ---")

# 1. Buka koneksi ke MongoDB lokal
client = MongoClient('mongodb://localhost:27017/')
db = client['cookcash_db']
collection = db['resep']

# 2. Tarik semua data dan jadikan tabel (DataFrame)
df_train = pd.DataFrame(list(collection.find()))
if '_id' in df_train.columns:
    df_train.drop(columns=['_id'], inplace=True) # Hapus ID bawaan Mongo

# 3. Bikin 'Buku Pintar': Gabungin Judul dan Bahan
df_train['Teks_Gabungan'] = df_train['Title Cleaned'].astype(str) + " " + df_train['Ingredients Cleaned'].astype(str)
df_train['Teks_Gabungan'] = df_train['Teks_Gabungan'].fillna('')

print(f" Siap! {len(df_train)} resep sudah ditarik dari MongoDB.")

--- TAHAP 1: AMBIL DATA DARI MONGODB ---
 Siap! 9275 resep sudah ditarik dari MongoDB.


In [2]:
print("--- TAHAP 2: PEMODELAN TF-IDF ---")

# 1. Panggil fungsi penerjemah
tfidf = TfidfVectorizer()

# 2. Ajari komputer mengubah 'Buku Pintar' teks menjadi angka (Matriks)
tfidf_matrix = tfidf.fit_transform(df_train['Teks_Gabungan'])

print("Model selesai dilatih dan diubah jadi angka!")

--- TAHAP 2: PEMODELAN TF-IDF ---
Model selesai dilatih dan diubah jadi angka!


In [3]:
print("--- TAHAP 3: SIMPAN MODEL ---")

# Simpan alat penerjemah (tfidf), otak angka (tfidf_matrix), dan data aslinya (df_train)
with open('model_cookcash.pkl', 'wb') as f:
    pickle.dump((tfidf, tfidf_matrix, df_train), f)

print(" Model sukses disimpan jadi file 'model_cookcash.pkl'.")

--- TAHAP 3: SIMPAN MODEL ---
 Model sukses disimpan jadi file 'model_cookcash.pkl'.
